# Train Spectron MoE trên Colab T4 (Vulkan, Tensor Core)

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từ trên xuống. Notebook cài Nim 2.2.10 và Vulkan, clone repo kèm submodule, tải corpus 5 tỷ token (`make download-spectron-scale-data`, khoảng 6,25 GB), build trainer `test_nimformer` với `-d:backend=vulkan`, rồi train MoE.

- **Tensor Core:** trainer chạy với `--tensor-cores=require`. Lúc khởi động, trainer chạy thử một GEMM 64×64 qua `KHR_cooperative_matrix` (đầu vào FP16, cộng dồn FP32) rồi so với kết quả đúng. Nếu driver không hỗ trợ, trainer dừng ngay thay vì âm thầm chạy FP32. Mỗi dòng log `[tensor-core]` đếm số GEMM chạy trên Tensor Core và số GEMM nhỏ chạy FP32.
- **Batch:** cell chọn batch chạy thử 3 bước cho mỗi kích thước, đo tok/s sau bước khởi động, rồi giữ batch nhanh nhất mà T4 và RAM của Colab chạy được. Đặt `BATCH` thành một số để bỏ qua bước đo.
- **Checkpoint:** cứ mỗi 20 bước, trainer lưu `test/spectron/models/moe/finetune.nimq` và upload lên [bobbydeveloper2014/bybyai-spectron](https://huggingface.co/bobbydeveloper2014/bybyai-spectron) tại `moe/finetune.nimq`. Upload chạy nền nên không dừng train. Nếu upload chưa xong mà đã có bản lưu mới, bản cũ đang chờ bị bỏ và chỉ bản mới nhất được upload. Khi Colab ngắt kết nối, chạy lại các cell từ đầu: cell tải dữ liệu sẽ kéo checkpoint mới nhất từ Hugging Face về, và trainer train tiếp cùng trạng thái Adam.

Cần một token Hugging Face có quyền ghi. Thêm token vào **Colab Secrets** (biểu tượng chìa khóa ở thanh bên trái) với tên `HF_TOKEN` và bật quyền truy cập cho notebook.

In [ ]:
# 1) Kiểm tra runtime GPU và cài các gói hệ thống, gồm Vulkan runtime và shader compiler (glslc).
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'Notebook này cần chạy trên Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('Chưa có GPU NVIDIA. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','pkg-config','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)

In [ ]:
# 2) Cài Nim 2.2.10 chính chủ và xác minh checksum (tokenizer thuần Nim).
import hashlib, urllib.request, tarfile, os, subprocess
from pathlib import Path

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'

# Tải file với User-Agent giả lập trình duyệt để tránh lỗi 403
if not archive.exists():
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(archive, 'wb') as out_file:
        out_file.write(response.read())

# Xác minh checksum
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'

# Giải nén
with tarfile.open(archive, 'r:xz') as tf:
    tf.extractall(TOOLS, filter='data')

# Cấu hình PATH
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:" + os.environ['PATH']

# Kiểm tra phiên bản
subprocess.run(['nim', '--version'], check=True)

In [ ]:
# 3) Clone repository bản main mới nhất và tải submodule đã ghim trong repo.
import os, subprocess
from pathlib import Path

URL = 'https://github.com/bobbyshop-vui/nimformers-framework.git'
BRANCH = 'main'
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'sync', '--recursive'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO, '| branch:', BRANCH)
print(subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))

In [ ]:
# 4) Cài Vulkan + kiểm tra ICD NVIDIA và xác nhận dGPU (không chấp nhận lavapipe/CPU)
import subprocess, os, re
from pathlib import Path

# --- Bước 1: Cài vulkan-tools ---
print("[1/4] Cài vulkan-tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)

# --- Bước 2: Xác định phiên bản driver NVIDIA và cài libnvidia-gl tương ứng ---
print("[2/4] Phát hiện phiên bản driver NVIDIA...")
smi = subprocess.check_output(['nvidia-smi'], text=True)
m = re.search(r'Driver Version:\s*(\d+)\.', smi)
assert m, "Không đọc được Driver Version từ nvidia-smi"
driver_major = m.group(1)
print(f"    -> Driver major: {driver_major}")

pkg = f'libnvidia-gl-{driver_major}'
print(f"[2/4] Cài {pkg}...")
subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], check=True)

# --- Bước 3: Đảm bảo file ICD NVIDIA có thể được loader tìm thấy ---
print("[3/4] Cấu hình Vulkan ICD...")
Path('/usr/share/vulkan/icd.d').mkdir(parents=True, exist_ok=True)
candidates = [
    '/etc/vulkan/icd.d/nvidia_icd.json',
    '/usr/share/vulkan/icd.d/nvidia_icd.json',
    f'/usr/lib/x86_64-linux-gnu/libnvidia-gl-{driver_major}/nvidia_icd.json',
]
src = next((c for c in candidates if Path(c).exists()), None)
if src and src != '/usr/share/vulkan/icd.d/nvidia_icd.json':
    subprocess.run(['ln', '-sf', src, '/usr/share/vulkan/icd.d/nvidia_icd.json'], check=True)
    print(f"    -> symlink: {src} -> /usr/share/vulkan/icd.d/nvidia_icd.json")
else:
    print(f"    -> ICD có sẵn: {src}")

# --- Bước 4: Chạy kiểm tra ---
print("[4/4] Kiểm tra Vulkan...")
subprocess.run(['vulkaninfo', '--summary'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo', '--summary'],
                                   text=True, stderr=subprocess.STDOUT)
print(''.join(line for line in summary.splitlines()
              if any(k in line.lower() for k in ('device name', 'device type', 'driver name'))))

assert 'NVIDIA' in summary.upper(), \
    'Không thấy NVIDIA Vulkan ICD. Runtime GPU hiện tại không expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), \
    'Không tìm thấy NVIDIA dGPU qua Vulkan; dừng để tránh fallback CPU/iGPU.'

print("✅ Vulkan OK: NVIDIA dGPU đã được expose.")

In [ ]:
# 5) Run settings and the Hugging Face write token from Colab Secrets.
import os
from pathlib import Path
from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as error:
    raise RuntimeError('Add a Hugging Face write token named HF_TOKEN in Colab Secrets.') from error

os.environ['SPECTRON_HF_REPO'] = 'bobbydeveloper2014/bybyai-spectron'
os.environ['SPECTRON_HF_CHECKPOINT'] = 'moe/finetune.nimq'

DATASET = REPO / 'test/spectron/data/scale5b'
CHECKPOINT = REPO / 'test/spectron/models/moe/finetune.nimq'
HOOK = REPO / 'test/spectron/hf_checkpoint_sync.py'
TRAINER = REPO / 'test/spectron/test_nimformer'
EXPERTS = 2         # Each expert has 29,985,344 parameters.
SEQ = 128
SAVE_EVERY = 20     # Save and upload every 20 optimizer steps.
STEPS = 100_000     # Additional steps for this session; rerun to continue.
BATCH = None        # None measures the fastest batch below; set a number to skip it.
print('Checkpoint:', CHECKPOINT, '->', os.environ['SPECTRON_HF_REPO'] + '/' + os.environ['SPECTRON_HF_CHECKPOINT'])

In [ ]:
# 6) Download the packed corpus (about 6.25 GB) and the latest checkpoint from Hugging Face.
import os
os.chdir(str(REPO))
!make download-spectron-scale-data
!python3 "{HOOK}" --restore "{CHECKPOINT}"

In [ ]:
# 7) Compile the Vulkan shaders and the Spectron trainer for the Vulkan backend.
import os
os.chdir(str(REPO))
!make shaders
!make build-spectron-trainer SPECTRON_TRAIN_BACKEND=vulkan
assert TRAINER.is_file(), f'Trainer not found: {TRAINER}'

In [ ]:
# 8) Measure tok/s for each batch size and keep the fastest one that fits.
import re
import shutil
import subprocess
from pathlib import Path


def run_streaming(command):
    # Stream combined output and return (exit code, text).
    process = subprocess.Popen([str(part) for part in command], cwd=str(REPO),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, errors='replace')
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    return process.wait(), ''.join(lines)


def measure(batch):
    probe_dir = Path('/content/spectron-batch-probe')
    shutil.rmtree(probe_dir, ignore_errors=True)
    code, text = run_streaming([
        TRAINER, f'--dataset={DATASET}', f'--checkpoint={probe_dir}/probe.nimq',
        f'--moe-experts={EXPERTS}', f'--batch={batch}', f'--seq={SEQ}', '--steps=3',
        '--log-every=1000', '--eval-batches=1', '--tensor-cores=require',
    ])
    shutil.rmtree(probe_dir, ignore_errors=True)
    # The step-3 line averages steps 2 and 3, after shader compilation in step 1.
    rates = re.findall(r'\[train\] step=3/\d+ .*? tok/s=(\d+)', text)
    if code != 0 or not rates:
        print(f'[batch-probe] batch={batch} failed with exit code {code}')
        return None
    return int(rates[-1])


if BATCH is None:
    results = {}
    for batch in (16, 32, 64, 96, 128, 192, 256):
        rate = measure(batch)
        if rate is None:
            break
        results[batch] = rate
        print(f'[batch-probe] batch={batch} tok/s={rate}')
        # Stop once a larger batch no longer beats the best smaller batch by 2%.
        if len(results) >= 2 and rate < 1.02 * max(r for b, r in results.items() if b != batch):
            break
    assert results, 'No batch size completed a training step.'
    BATCH = max(results, key=results.get)
    print('[batch-probe] results:', results)
print('BATCH =', BATCH)

In [ ]:
# 9) Train. Every SAVE_EVERY steps the checkpoint is saved and uploaded in the background.
import os
import signal
import subprocess
import sys

os.chdir(str(REPO))
command = [
    TRAINER, f'--dataset={DATASET}', f'--checkpoint={CHECKPOINT}',
    f'--moe-experts={EXPERTS}', f'--batch={BATCH}', f'--seq={SEQ}', f'--steps={STEPS}',
    f'--checkpoint-every={SAVE_EVERY}', f'--log-every={SAVE_EVERY}',
    f'--checkpoint-hook={HOOK}', '--tensor-cores=require',
]
print(' '.join(str(part) for part in command), flush=True)
process = subprocess.Popen([str(part) for part in command], cwd=str(REPO),
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, errors='replace')
try:
    # The upload worker shares this pipe, so the loop also shows its messages.
    for line in process.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    print('Stopping the trainer; the last saved checkpoint is kept.', flush=True)
    process.send_signal(signal.SIGTERM)
    for line in process.stdout:
        print(line, end='', flush=True)
finally:
    code = process.wait()
    subprocess.run([sys.executable, str(HOOK), '--wait', str(CHECKPOINT)], check=True)
    print('Trainer exit code:', code)

### Ghi chú

- Log `[train]` ghi `step`, `epoch`, `loss`, `tok/s` và số token đã train. Log `[val]` ghi loss validation của code và tiếng Anh mỗi 500 bước. Log `[hf-sync]` cho biết bước nào đã được đưa vào hàng đợi và đã upload xong.
- Mỗi lần upload là một commit trên Hugging Face chứa toàn bộ checkpoint, gồm trọng số, router và trạng thái Adam của từng expert.
- Đổi `EXPERTS` thành số lớn hơn để tăng số expert của checkpoint hiện có. Trainer thêm expert mới và train tiếp cùng router.

### Tài liệu

- [Hướng dẫn train Spectron MoE](https://github.com/bobbyshop-vui/nimformers-framework/blob/main/docs/spectron-guide.vi.md)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [VK_KHR_cooperative_matrix](https://registry.khronos.org/vulkan/specs/latest/man/html/VK_KHR_cooperative_matrix.html)